In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## **RAG Eval Recipe**

This Eval Recipe demonstrates how to compare performance of two models on a RAG dataset using [Vertex AI Evaluation Service](https://cloud.google.com/vertex-ai/generative-ai/docs/models/evaluation-overview).

<table align="left">
  <td style="text-align: center">
    <a href="https://art-analytics.appspot.com/r.html?uaid=G-FHXEFWTT4E&utm_source=aRT-eval-recipe-rag&utm_medium=aRT-clicks&utm_campaign=eval-recipe-rag&destination=eval-recipe-rag&url=https%3A%2F%2Fcolab.research.google.com%2Fgithub%2FGoogleCloudPlatform%2Fapplied-ai-engineering-samples%2Fblob%2Fmain%2Fgenai-on-vertex-ai%2Fgemini%2Fmodel_upgrades%2Frag%2Fvertex_colab%2Frag_eval.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://art-analytics.appspot.com/r.html?uaid=G-FHXEFWTT4E&utm_source=aRT-eval-recipe-rag&utm_medium=aRT-clicks&utm_campaign=eval-recipe-rag&destination=eval-recipe-rag&url=https%3A%2F%2Fconsole.cloud.google.com%2Fvertex-ai%2Fcolab%2Fimport%2Fhttps%3A%252F%252Fraw.githubusercontent.com%252FGoogleCloudPlatform%252Fapplied-ai-engineering-samples%252Fmain%252Fgenai-on-vertex-ai%252Fgemini%252Fmodel_upgrades%252Frag%252Fvertex_colab%252Frag_eval.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://art-analytics.appspot.com/r.html?uaid=G-FHXEFWTT4E&utm_source=aRT-eval-recipe-rag&utm_medium=aRT-clicks&utm_campaign=eval-recipe-rag&destination=eval-recipe-rag&url=https%3A%2F%2Fconsole.cloud.google.com%2Fvertex-ai%2Fworkbench%2Fdeploy-notebook%3Fdownload_url%3Dhttps%3A%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fapplied-ai-engineering-samples%2Fmain%2Fgenai-on-vertex-ai%2Fgemini%2Fmodel_upgrades%2Frag%2Fvertex_colab%2Frag_eval.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/applied-ai-engineering-samples/blob/main/genai-on-vertex-ai/gemini/model_upgrades/rag/vertex_colab/rag_eval.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

- Use case: answering questions based on information from the specified source documents(RAG).

- Metric: This eval uses BLEU and ROUGE to rate Question Answering Quality.

- Evaluation Dataset is based on [Single-Topic RAG Evaluation Dataset](https://www.kaggle.com/datasets/samuelmatsuoharris/single-topic-rag-evaluation-dataset/). It includes 4 randomly sampled documents stored as plain text files, and a JSONL file `dataset.jsonl` that provides ground truth answers. Each record in this file includes 3 attributes:
    - `document_uri`: relative path to the plain text document file
    - `question`: the question that we want to ask about this particular document
    - `reference`: expected correct answer from the document.

- Prompt Template is a zero-shot prompt located in [`prompt_template.txt`](./prompt_template.txt) with two prompt variables (`document_text` and `question`) that are automatically populated from our dataset.


### Configure Eval Settings

In [ ]:
%%writefile .env
PROJECT_ID=your-poject-id           # Google Cloud Project ID
LOCATION=us-central1                  # Region for all required Google Cloud services
EXPERIMENT_NAME=rag-eval-recipe-demo      # Creates Vertex AI Experiment to track the eval runs
MODEL_BASELINE=gemini-1.0-pro-001   # Name of your current model
MODEL_CANDIDATE=gemini-2.0-flash # This model will be compared to the baseline model
DATASET_URI="gs://gemini_migration_assets/rag_vertex/dataset.jsonl"  # Evaluation dataset in Google Cloud Storage
PROMPT_TEMPLATE_URI="gs://gemini_migration_assets/rag_vertex/prompt_template.txt"  # Text file in Google Cloud Storage

### Install Python Libraries

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform[evaluation] plotly python-dotenv
# The error "session crashed" is expected. Please ignore it and proceed to the next cell.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import os
import json
import pandas as pd
import sys
import vertexai
from dotenv import load_dotenv
from google.cloud import storage

from datetime import datetime
from IPython.display import clear_output
from vertexai.evaluation import EvalTask, EvalResult
from vertexai.generative_models import GenerativeModel

### Authenticate to Google Cloud (requires permission to open a popup window)

In [ ]:
load_dotenv(override=True)
if os.getenv("PROJECT_ID") == "your-project-id":
    raise ValueError("Please configure your Google Cloud Project ID in the first cell.")
if "google.colab" in sys.modules:  
    from google.colab import auth  
    auth.authenticate_user()
vertexai.init(project=os.getenv('PROJECT_ID'), location=os.getenv('LOCATION'))

### Run the eval on both models and compare BLEU and ROUGE metrics 

In [ ]:
metrics = [
    "rouge_l_sum",
    "bleu",
]

def load_file(gcs_uri: str) -> str:
    blob = storage.Blob.from_string(gcs_uri, storage.Client())
    return blob.download_as_string().decode('utf-8')

def load_dataset(dataset_uri: str):
    jsonl = load_file(dataset_uri)
    samples = [json.loads(line) for line in jsonl.splitlines() if line.strip()]
    df = pd.DataFrame(samples)
    df['document_text'] = df['document_uri'].apply(lambda document_uri: load_file(document_uri))
    return df[['question', 'reference', 'document_text']]

def load_prompt_template() -> str:
    blob = storage.Blob.from_string(os.getenv("PROMPT_TEMPLATE_URI"), storage.Client())
    return blob.download_as_string().decode('utf-8')

def run_eval(model: str) -> EvalResult:
  timestamp = f"{datetime.now().strftime('%b-%d-%H-%M-%S')}".lower()
  return EvalTask(
      dataset=os.getenv("DATASET_URI"),
      metrics=metrics,
      experiment=os.getenv('EXPERIMENT_NAME')
  ).evaluate(
      model=GenerativeModel(model),
      prompt_template=load_prompt_template(),
      experiment_run_name=f"{timestamp}-{model.replace('.', '-')}"
  )

baseline = run_eval(os.getenv("MODEL_BASELINE"))
candidate = run_eval(os.getenv("MODEL_CANDIDATE"))
clear_output()
print("Baseline model Rouge:", round(baseline.summary_metrics['rouge_l_sum/mean'],3))
print("Canidadate model Rouge:", round(candidate.summary_metrics['rouge_l_sum/mean'],3))
print("Baseline model Bleu:", round(baseline.summary_metrics['bleu/mean'],3))
print("Candidate model Bleu:", round(candidate.summary_metrics['bleu/mean'],3))